# 🧬 Genomic Attribution Visualizer

A bioinformatics pipeline that bridges ML model interpretability and genomic context.  
Takes a **3D attribution tensor** `(samples × SNPs × classes)` and a **BED file** of SNP coordinates,  
and produces:

- 📊 Per-chromosome attribution profiles (Gaussian-smoothed, dark background)
- 📍 Statistically significant peak detection (**FDR** & **IDR**)
- 🔬 Interactive **IGV-like browser** with GENCODE gene annotation track

**Assembly:** GRCh38 | **Annotation:** [GENCODE](https://www.gencodegenes.org/human/)

---

## 📁 Directory structure

Place **exactly one file** in each input folder before running:

```
project/
├── genomic_attribution_visualizer.ipynb
├── tensor/          ← your .npy tensor  (samples × SNPs × classes)
├── bed_file/        ← your .bed file    (chrom  start  end, no header)
├── annotation/      ← gencode_genes.gtf (gene rows only — see below)
└── chr_plots/       ← output PNGs (auto-created)
```

### Gene annotation (one-time setup)
```bash
wget https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_47/gencode.v47.annotation.gtf.gz
zcat gencode.v47.annotation.gtf.gz | awk '$3 == "gene"' > annotation/gencode_genes.gtf
```

### Install dependencies
```bash
pip install numpy pandas scipy matplotlib statsmodels ipywidgets
```


## ⚙️ Section 0 — User Configuration

> **This is the only cell you need to edit.**  
> Put your files in the correct folders, then adjust class names, colors, and parameters below.


In [ ]:
import os

# ── Helper: auto-detect the single file inside a folder ────────────────────
def _first(folder):
    """Return the path to the only file in `folder` (ignores hidden files).
    Raises a clear error if the folder is empty or contains multiple files."""
    os.makedirs(folder, exist_ok=True)
    files = [f for f in os.listdir(folder)
             if not f.startswith('.') and
             os.path.isfile(os.path.join(folder, f))]
    if not files:
        raise FileNotFoundError(
            f"No file found in '{folder}'. "
            f"Please place exactly one file there.")
    if len(files) > 1:
        raise ValueError(
            f"Multiple files found in '{folder}': {files}. "
            f"Leave only one.")
    return os.path.join(folder, files[0])

# ============================================================
#  INPUT FILES — put exactly one file in each folder
# ============================================================
TENSOR_PATH = _first('./tensor/*.npy')       # .npy, shape: (samples, SNPs, classes)
BED_PATH    = _first('./bed_file/*.bed')     # BED, no header: chrom  start  end
GTF_PATH    = _first('./annotation/*.gtf')   # GTF with gene rows only
CLASS_PATH  = _first('./annotation/*.tsv')

# ============================================================
#  OUTPUT
# ============================================================
PLOTS_DIR = './chr_plots'

# ============================================================
#  CLASS LABELS & COLORS
#  → Add or remove entries to match the number of classes
#    in your tensor (tensor.shape[2]).
#  → Any valid matplotlib color string or hex code works.
# ============================================================


# ============================================================
#  SMOOTHING
# ============================================================
GRID_STEP = 100_000   # Genomic grid resolution in bp (100 kb recommended)
SIGMA     = 7         # Gaussian sigma (grid units); increase for smoother profiles

# ============================================================
#  PEAK DETECTION
# ============================================================
FDR_ALPHA         = 0.05   # Benjamini-Hochberg significance threshold
FDR_HEIGHT_MULT   = 0.5    # Peak height = mean + mult × std
IDR_HEIGHT_MULT   = 0.3    # Same for IDR candidates
IDR_SCALE_MIN_KB  = 80     # Smallest IDR window (kb)
IDR_SCALE_MAX_MB  = 10     # Largest  IDR window (Mb)
IDR_SCALE_STEP_KB = 500    # Step between IDR window sizes (kb)
IDR_MIN_FRACTION  = 0.3    # Min fraction of scales where peak must be top-ranked
IDR_RANK_THRESH   = 0.80   # Normalised rank threshold

# ============================================================
#  IGV BROWSER
# ============================================================
IGV_DEFAULT_WINDOW = 5_000_000   # Default view width in bp

# ============================================================
#  CHROMOSOME ORDER  (GRCh38 — modify only if needed)
# ============================================================
CHR_ORDER = [f'chr{i}' for i in range(1, 23)] + ['chrX', 'chrY', 'chrM']

# ── Print summary ───────────────────────────────────────────────────────────
print("Configuration loaded successfully.")
print(f"  Tensor     : {TENSOR_PATH}")
print(f"  BED        : {BED_PATH}")
print(f"  GTF        : {GTF_PATH}")
print(f"  Classes    : {CLASS_NAMES}")


## 📦 Section 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d
from scipy.signal import find_peaks
from scipy.stats import zscore, norm
from statsmodels.stats.multitest import multipletests
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings

warnings.filterwarnings('ignore')
os.makedirs(PLOTS_DIR, exist_ok=True)

with open(CLASS_PATH, "r") as f:
    CLASS_NAMES = [line.strip() for line in f]
N_CLASSES = len(CLASS_NAMES)

def get_class_colors(num_classes):
    if num_classes <= 10:
        cmap = plt.cm.tab10
    elif num_classes <= 20:
        cmap = plt.cm.tab20
    else:
        cmap = plt.cm.gist_rainbow
    
    colors = [cmap(i / num_classes) for i in range(num_classes)]
    class_colors = [matplotlib.colors.rgb2hex(color) for color in colors]
    
    return class_colors
CLASS_COLORS =  get_class_colors(N_CLASSES)

# ── Validate config ─────────────────────────────────────────────────────────
assert len(CLASS_COLORS) == N_CLASSES, (
    f"CLASS_COLORS has {len(CLASS_COLORS)} entries but CLASS_NAMES has {N_CLASSES}. "
    "Please fix Section 0.")

# Auto-generate colors for any extra classes not covered by CLASS_COLORS
import matplotlib.cm as cm
def get_color(cls):
    if cls < len(CLASS_COLORS):
        return CLASS_COLORS[cls]
    cmap = cm.get_cmap('tab20')
    return matplotlib.colors.to_hex(cmap(cls % 20))

print(f"Imports OK | {N_CLASSES} class(es) detected")


## 1️⃣ Load & Preprocess Attribution Tensor

In [ ]:
tensor_raw = np.load(TENSOR_PATH)
print(f"Tensor shape : {tensor_raw.shape}   (samples × SNPs × classes)")
print(f"Value range  : [{tensor_raw.min():.4f},  {tensor_raw.max():.4f}]")

assert tensor_raw.ndim == 3, (
    f"Expected a 3-D tensor (samples, SNPs, classes), got shape {tensor_raw.shape}")

# Auto-detect number of classes from tensor if user left CLASS_NAMES default
n_cls_tensor = tensor_raw.shape[2]
if n_cls_tensor != N_CLASSES:
    print(f"\n⚠️  Tensor has {n_cls_tensor} classes but CLASS_NAMES defines {N_CLASSES}.")
    print("   Auto-generating class labels and colors…")
    CLASS_NAMES  = [f'Class {i}' for i in range(n_cls_tensor)]
    CLASS_COLORS = [get_color(i) for i in range(n_cls_tensor)]
    N_CLASSES    = n_cls_tensor
    print(f"   New labels : {CLASS_NAMES}")
    print(f"   New colors : {CLASS_COLORS}")

# Subtract baseline (1.0 for multiplicative attributions, e.g. raw IG scores)
tensor       = tensor_raw - 1.0
median_attrs = np.median(tensor, axis=0)   # shape: (SNPs, classes)

print(f"\nAfter subtracting 1 : [{tensor.min():.4f},  {tensor.max():.4f}]")
print(f"Median tensor shape  : {median_attrs.shape}  (SNPs × classes)")


## 2️⃣ Load & Sort BED File

In [ ]:
bed = pd.read_csv(BED_PATH, sep='\t', header=True)
bed['snp_idx'] = np.arange(len(bed))
print(f"BED loaded : {len(bed):,} SNPs")

assert len(bed) == tensor_raw.shape[1], (
    f"BED has {len(bed)} rows but tensor has {tensor_raw.shape[1]} SNP positions. "
    "They must match exactly.")

unknown = set(bed['chrom'].unique()) - set(CHR_ORDER)
if unknown:
    print(f"  ⚠️  Chromosomes not in CHR_ORDER (will be skipped): {sorted(unknown)}")

bed['chrom_cat'] = pd.Categorical(bed['chrom'], categories=CHR_ORDER, ordered=True)
bed_sorted       = bed.sort_values(['chrom_cat', 'start']).reset_index(drop=True)

sorted_indices = bed_sorted['snp_idx'].values
median_sorted  = median_attrs[sorted_indices, :]

chroms_present = [c for c in CHR_ORDER if c in bed_sorted['chrom'].values]
print(f"Chromosomes : {len(chroms_present)} found")
for c in chroms_present:
    n = (bed_sorted['chrom'] == c).sum()
    print(f"  {c:8s}: {n:5d} SNPs")


## 3️⃣ Smooth Profiles (Gaussian smoothing and linear interpolation)

In [ ]:
def smooth_genomic(positions, values, grid_step=GRID_STEP, sigma=SIGMA):
    """
    Project SNP-level signal onto a uniform genomic grid, then apply
    Gaussian smoothing so that sigma is physically uniform across the
    chromosome regardless of local SNP density.

    Parameters
    ----------
    positions : array-like  — sorted SNP start coordinates (bp)
    values    : array-like  — attribution values (same length)
    grid_step : int         — grid resolution in bp
    sigma     : float       — Gaussian sigma in grid units

    Returns
    -------
    grid     : ndarray — uniform position array
    smoothed : ndarray — smoothed signal on the grid
    """
    grid     = np.arange(positions[0], positions[-1] + grid_step,
                         grid_step, dtype=float)
    interp   = interp1d(positions, values, kind='linear',
                        bounds_error=False, fill_value='extrapolate')
    smoothed = gaussian_filter1d(interp(grid), sigma=sigma)
    return grid, smoothed


chr_data = {}
for chrom in chroms_present:
    mask      = bed_sorted['chrom'] == chrom
    positions = bed_sorted.loc[mask, 'start'].values
    raw_vals  = median_sorted[mask.values, :]

    grid_ref, smoothed_list = None, []
    for cls in range(N_CLASSES):
        grid, sm = smooth_genomic(positions, raw_vals[:, cls])
        smoothed_list.append(sm)
        grid_ref = grid

    chr_data[chrom] = {
        'positions': positions,
        'grid':      grid_ref,
        'raw':       raw_vals,
        'smoothed':  np.stack(smoothed_list, axis=1),   # (grid_pts × N_CLASSES)
    }

print(f"Processed {len(chr_data)} chromosome(s).")


## 4️⃣ Per-Chromosome Attribution Profiles

In [ ]:
def plot_chromosome(chrom, data, save_path=None, figsize=(16, 5)):
    """Plot smoothed attribution profiles for all classes on one chromosome."""
    grid     = data['grid']
    smoothed = data['smoothed']

    fig, ax = plt.subplots(figsize=figsize, facecolor='black')
    ax.set_facecolor('black')

    for cls in range(N_CLASSES):
        y   = smoothed[:, cls]
        col = CLASS_COLORS[cls]
        ax.plot(grid, y, color=col, linewidth=1.3,
                label=CLASS_NAMES[cls], zorder=3)
        ax.fill_between(grid, 0, y, where=(y > 0), color=col, alpha=0.18, zorder=2)
        ax.fill_between(grid, 0, y, where=(y < 0), color=col, alpha=0.18, zorder=2)

    ax.axhline(0, color='#555555', linewidth=0.7, linestyle='--', zorder=1)
    ax.set_title(chrom, color='white', fontsize=14, fontweight='bold', pad=8)
    ax.set_xlabel('Genomic position', color='#AAAAAA', fontsize=10)
    ax.set_ylabel('Attribution (median − 1)', color='#AAAAAA', fontsize=10)
    ax.tick_params(colors='#AAAAAA', labelsize=8)
    for sp in ax.spines.values():
        sp.set_edgecolor('#444444')
    ax.legend(facecolor='#111111', edgecolor='#444444', labelcolor='white',
              fontsize=8, loc='upper right', framealpha=0.8,
              ncol=max(1, N_CLASSES // 6))
    ax.xaxis.set_major_formatter(
        matplotlib.ticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f} Mb'))

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, facecolor='black', bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()


for chrom in chroms_present:
    plot_chromosome(chrom, chr_data[chrom],
                    save_path=os.path.join(PLOTS_DIR, f'{chrom}.png'))
    print(f"  {chrom} ✓")

print(f"\nAll plots saved to '{PLOTS_DIR}/'")


## 5️⃣ Significant Peak Detection

### FDR (Benjamini-Hochberg)

In [ ]:
def detect_peaks_fdr(positions, signal,
                     alpha=FDR_ALPHA,
                     height_std_mult=FDR_HEIGHT_MULT,
                     min_distance=3):
    """
    Detect significant peaks using FDR correction (Benjamini-Hochberg).

    Steps:
      1. Find peak candidates via scipy.signal.find_peaks
      2. Compute z-scores relative to the full signal
      3. Convert to two-tailed p-values
      4. Apply BH correction at level `alpha`
    """
    if len(signal) < 5:
        return np.array([], dtype=int)
    peak_idxs, _ = find_peaks(
        signal,
        height=signal.mean() + height_std_mult * signal.std(),
        distance=min_distance)
    if len(peak_idxs) == 0:
        return np.array([], dtype=int)
    zs    = np.abs(zscore(signal))[peak_idxs]
    pvals = np.clip(2 * (1 - norm.cdf(zs)), 1e-300, 1.0)
    reject, _, _, _ = multipletests(pvals, alpha=alpha, method='fdr_bh')
    return peak_idxs[reject]


### IDR (Multi-Scale Rank Stability)

In [ ]:
def detect_peaks_idr(positions, signal,
                     scale_min_kb=IDR_SCALE_MIN_KB,
                     scale_max_mb=IDR_SCALE_MAX_MB,
                     scale_step_kb=IDR_SCALE_STEP_KB,
                     min_scales_fraction=IDR_MIN_FRACTION,
                     height_std_mult=IDR_HEIGHT_MULT,
                     min_distance=3):
    """
    Multi-scale rank-stability peak detection (IDR-inspired).

    For each candidate peak, compute its normalised rank within windows
    of increasing genomic size. A peak is significant if it ranks in the
    top (1 - IDR_RANK_THRESH) fraction on at least `min_scales_fraction`
    of all tested scales — rewarding peaks that dominate at multiple
    resolutions rather than isolated noise spikes.
    """
    if len(signal) < 5:
        return np.array([], dtype=int)
    scales_bp = np.arange(scale_min_kb * 1_000,
                          scale_max_mb  * 1_000_000 + 1,
                          scale_step_kb * 1_000)
    peak_idxs, _ = find_peaks(
        signal,
        height=signal.mean() + height_std_mult * signal.std(),
        distance=min_distance)
    if len(peak_idxs) == 0:
        return np.array([], dtype=int)

    n_scales    = len(scales_bp)
    rank_matrix = np.zeros((len(peak_idxs), n_scales))

    for si, scale in enumerate(scales_bp):
        half = scale / 2
        for pi, pidx in enumerate(peak_idxs):
            in_win      = ((positions >= positions[pidx] - half) &
                           (positions <= positions[pidx] + half))
            window_vals = signal[in_win]
            if len(window_vals) < 2:
                continue
            rank_matrix[pi, si] = (
                (window_vals < signal[pidx]).sum() + 1
            ) / len(window_vals)

    scales_above = (rank_matrix >= IDR_RANK_THRESH).sum(axis=1)
    min_n        = int(np.ceil(min_scales_fraction * n_scales))
    return peak_idxs[scales_above >= min_n]


### Run detection & compare

In [ ]:
peaks_fdr = {c: {} for c in chroms_present}
peaks_idr = {c: {} for c in chroms_present}

print("Running peak detection…")
for chrom in chroms_present:
    grid     = chr_data[chrom]['grid']
    smoothed = chr_data[chrom]['smoothed']
    for cls in range(N_CLASSES):
        sig = smoothed[:, cls]
        peaks_fdr[chrom][cls] = detect_peaks_fdr(grid, sig)
        peaks_idr[chrom][cls] = detect_peaks_idr(grid, sig)
    print(f"  {chrom} ✓")

print("Done.")


In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────
records = []
for chrom in chroms_present:
    row = {'Chromosome': chrom}
    tf = ti = 0
    for cls in range(N_CLASSES):
        nf = len(peaks_fdr[chrom][cls])
        ni = len(peaks_idr[chrom][cls])
        row[f'FDR {CLASS_NAMES[cls]}'] = nf
        row[f'IDR {CLASS_NAMES[cls]}'] = ni
        tf += nf; ti += ni
    row['FDR Total'] = tf
    row['IDR Total'] = ti
    records.append(row)

df_peaks  = pd.DataFrame(records).set_index('Chromosome')
grand_fdr = df_peaks['FDR Total'].sum()
grand_idr = df_peaks['IDR Total'].sum()

print(f"Significant peaks — FDR : {grand_fdr}")
print(f"Significant peaks — IDR : {grand_idr}")
display(df_peaks)

best_method = 'FDR' if grand_fdr >= grand_idr else 'IDR'
peaks_best  = peaks_fdr if best_method == 'FDR' else peaks_idr
print(f"\n→ Selected for IGV: {best_method} ({max(grand_fdr, grand_idr)} peaks)")


## 6️⃣ Load Gene Annotation

In [ ]:
def parse_gtf_genes(gtf_path):
    """Parse GTF and return a DataFrame of gene-level records."""
    records = []
    with open(gtf_path) as fh:
        for line in fh:
            if line.startswith('#'):
                continue
            parts = line.rstrip().split('\t')
            if len(parts) < 9 or parts[2] != 'gene':
                continue
            chrom     = parts[0]
            start     = int(parts[3]) - 1   # 1-based → 0-based
            end       = int(parts[4])
            strand    = parts[6]
            gene_name = ''
            for attr in parts[8].split(';'):
                attr = attr.strip()
                if attr.startswith('gene_name'):
                    gene_name = (attr.split('"')[1]
                                 if '"' in attr else attr.split()[-1])
                    break
            records.append({'chrom': chrom, 'start': start, 'end': end,
                            'strand': strand, 'gene_name': gene_name})
    return pd.DataFrame(records)


print("Parsing GTF…")
genes_df = parse_gtf_genes(GTF_PATH)
print(f"Loaded {len(genes_df):,} gene records.")
display(genes_df.head())


## 7️⃣ Interactive IGV-like Browser

Use the controls to navigate:
- **Chromosome** — which chromosome to display
- **Window (bp)** — zoom level (500 kb → 50 Mb)
- **Start (bp)** — scroll position

Shaded areas mark **significant peaks** (winning method only).  
Gene track: 🔵 + strand · 🩷 − strand


In [ ]:
def draw_igv(chrom, view_start, view_end, peaks_dict, genes_df):
    """Render an IGV-like view for the given genomic window."""
    grid     = chr_data[chrom]['grid']
    smoothed = chr_data[chrom]['smoothed']

    mask  = (grid >= view_start) & (grid <= view_end)
    pos_w = grid[mask]
    sig_w = smoothed[mask, :]

    fig = plt.figure(figsize=(18, 7), facecolor='black')
    gs  = fig.add_gridspec(3, 1, height_ratios=[5, 1.5, 0.4], hspace=0.05)
    ax_main  = fig.add_subplot(gs[0])
    ax_genes = fig.add_subplot(gs[1], sharex=ax_main)
    ax_scale = fig.add_subplot(gs[2], sharex=ax_main)
    for ax in [ax_main, ax_genes, ax_scale]:
        ax.set_facecolor('black')

    # ── Attribution track ──────────────────────────────────────────────────
    local_indices = np.where(mask)[0]

    for cls in range(N_CLASSES):
        col  = CLASS_COLORS[cls]
        y_w  = sig_w[:, cls]
        ax_main.plot(pos_w, y_w, color=col, linewidth=1.3,
                     label=CLASS_NAMES[cls], zorder=3)

        sig_peaks    = peaks_dict[chrom].get(cls, np.array([], dtype=int))
        peaks_in_win = np.intersect1d(sig_peaks, local_indices)
        for pidx in peaks_in_win:
            wi_arr = np.where(local_indices == pidx)[0]
            if len(wi_arr) == 0:
                continue
            wi  = wi_arr[0]
            rgn = ((pos_w >= pos_w[wi] - GRID_STEP) &
                   (pos_w <= pos_w[wi] + GRID_STEP))
            ax_main.fill_between(pos_w[rgn], 0, y_w[rgn],
                                 color=col, alpha=0.45, zorder=2)

    ax_main.axhline(0, color='#555555', linewidth=0.7, linestyle='--', zorder=1)
    ax_main.set_ylabel('Attribution\n(median − 1)', color='#AAAAAA', fontsize=9)
    ax_main.tick_params(colors='#AAAAAA', labelsize=8, labelbottom=False)
    ax_main.set_title(
        f'{chrom}   {view_start/1e6:.3f} – {view_end/1e6:.3f} Mb'
        f'   [peak method: {best_method}]',
        color='white', fontsize=12, fontweight='bold', pad=6)
    for sp in ax_main.spines.values():
        sp.set_edgecolor('#333333')
    ax_main.legend(facecolor='#111111', edgecolor='#444444',
                   labelcolor='white', fontsize=8, loc='upper right',
                   framealpha=0.85, ncol=max(1, N_CLASSES // 6))

    # ── Gene track ─────────────────────────────────────────────────────────
    chrom_genes = genes_df[
        (genes_df['chrom'] == chrom) &
        (genes_df['end']   >= view_start) &
        (genes_df['start'] <= view_end)
    ]
    gene_y_levels = []
    for _, gene in chrom_genes.iterrows():
        g_start = max(gene['start'], view_start)
        g_end   = min(gene['end'],   view_end)
        width   = g_end - g_start
        if width <= 0:
            continue
        level = len(gene_y_levels)
        for lvl, last_end in enumerate(gene_y_levels):
            if g_start > last_end + 50_000:
                level = lvl
                gene_y_levels[lvl] = g_end
                break
        else:
            gene_y_levels.append(g_end)
        y_pos   = level * 0.25
        color_g = '#4FC3F7' if gene['strand'] == '+' else '#F48FB1'
        ax_genes.barh(y_pos, width, left=g_start, height=0.18,
                      color=color_g, alpha=0.8)
        if width > (view_end - view_start) * 0.01:
            ax_genes.text(g_start + width / 2, y_pos + 0.12,
                          gene['gene_name'], color='white',
                          fontsize=6, ha='center', va='bottom', clip_on=True)

    ax_genes.set_ylabel('Genes', color='#AAAAAA', fontsize=8)
    ax_genes.tick_params(colors='#AAAAAA', labelsize=7, labelbottom=False)
    ax_genes.set_ylim(-0.2, max(len(gene_y_levels) * 0.25 + 0.3, 0.6))
    for sp in ax_genes.spines.values():
        sp.set_edgecolor('#333333')
    ax_genes.legend(
        handles=[mpatches.Patch(color='#4FC3F7', label='+ strand'),
                 mpatches.Patch(color='#F48FB1', label='− strand')],
        facecolor='#111111', edgecolor='#444444', labelcolor='white',
        fontsize=7, loc='upper right', framealpha=0.85)

    # ── Position scale ─────────────────────────────────────────────────────
    ax_scale.set_facecolor('#0A0A0A')
    ax_scale.set_yticks([])
    span      = view_end - view_start
    raw_step  = span / 15
    mag       = 10 ** int(np.floor(np.log10(max(raw_step, 1))))
    nice_step = max(mag * round(raw_step / mag), 50)
    tick_pos  = np.arange(
        np.ceil(view_start / nice_step) * nice_step, view_end, nice_step)
    ax_scale.set_xticks(tick_pos)

    def fmt_pos(x):
        if x >= 1e6:   return f'{x/1e6:.3f} Mb'
        elif x >= 1e3: return f'{x/1e3:.1f} kb'
        else:          return f'{int(x)} bp'

    ax_scale.set_xticklabels([fmt_pos(t) for t in tick_pos],
                              color='#AAAAAA', fontsize=7,
                              rotation=45, ha='right')
    ax_scale.set_xlim(view_start, view_end)
    for sp in ax_scale.spines.values():
        sp.set_edgecolor('#333333')

    plt.tight_layout()
    plt.show()


print("draw_igv defined.")


In [ ]:
def build_igv_browser():
    """Launch the interactive IGV-like browser widget."""
    chr_ranges = {
        chrom: (int(chr_data[chrom]['grid'].min()),
                int(chr_data[chrom]['grid'].max()))
        for chrom in chroms_present
    }

    chr_selector = widgets.Dropdown(
        options=chroms_present, value=chroms_present[0],
        description='Chromosome:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='210px'))

    window_slider = widgets.IntSlider(
        value=IGV_DEFAULT_WINDOW,
        min=500_000, max=50_000_000, step=500_000,
        description='Window (bp):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='520px'))

    position_slider = widgets.IntSlider(
        value=chr_ranges[chroms_present[0]][0],
        min=chr_ranges[chroms_present[0]][0],
        max=chr_ranges[chroms_present[0]][1],
        step=500_000,
        description='Start (bp):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='520px'))

    out = widgets.Output()

    def update_chr(_):
        chrom = chr_selector.value
        cmin, cmax = chr_ranges[chrom]
        position_slider.min   = cmin
        position_slider.max   = cmax
        position_slider.value = cmin
        position_slider.step  = max((cmax - cmin) // 200, 50_000)
        refresh(None)

    def refresh(_):
        chrom      = chr_selector.value
        cmin, cmax = chr_ranges[chrom]
        view_start = max(position_slider.value, cmin)
        view_end   = min(view_start + window_slider.value, cmax)
        with out:
            clear_output(wait=True)
            draw_igv(chrom, view_start, view_end, peaks_best, genes_df)

    chr_selector.observe(update_chr, names='value')
    position_slider.observe(refresh, names='value')
    window_slider.observe(refresh,   names='value')

    display(widgets.VBox([
        widgets.HBox([chr_selector, window_slider]),
        position_slider
    ]), out)
    refresh(None)


build_igv_browser()
